In [15]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from dotenv import load_dotenv
import os

# PART 1 -Getting Started with Groq API

## Task 1: Groq API Setup & Basic Chat

In [3]:
load_dotenv()

True

In [6]:
Groq_api = os.getenv('GROQ_API_Key')

####  !pip install groq langchain-groq

In [10]:
llm = ChatGroq(model='llama-3.1-8b-instant')

In [11]:
Response = llm.invoke('Who are you ?')

In [13]:
Response.content

"I'm an artificial intelligence model known as a large language model (LLM) or a conversational AI. I'm a computer program designed to simulate human-like conversations, answer questions, and provide information on a wide range of topics.\n\nI was trained on a massive dataset of text from the internet, books, and other sources, which allows me to generate human-like responses to various prompts and questions. I can understand natural language, process it, and respond accordingly.\n\nI'm not a human, but rather a machine designed to assist and communicate with humans. I don't have personal opinions, emotions, or experiences, but I can provide information, answer questions, and engage in conversations to the best of my abilities based on my training data.\n\nI'm here to help you with any questions, topics, or tasks you'd like to discuss. Feel free to ask me anything, and I'll do my best to assist you!"

## Task 2: Build a Groq Chatbot (Core Logic)

In [14]:
def groq_chat(prompt :str):

    llm = ChatGroq(model='llama-3.1-8b-instant')
    response = llm.invoke(prompt)

    return response.content

In [18]:
prompt = ChatPromptTemplate.from_messages([('system','You are an Helpfulll AI Assistant'),('human','Who are you ?')])

In [17]:
input = 'Who are you ?'

In [20]:
groq_chat(input)

"I'm an artificial intelligence model known as a large language model (LLM) or a conversational AI. I'm a computer program designed to understand and generate human-like text responses to a wide range of questions and topics.\n\nI don't have a personal identity, and I'm not a human being. I exist solely to assist and provide information to users like you through text-based conversations. I can process and analyze vast amounts of data, learn from it, and generate responses based on that knowledge.\n\nMy primary function is to help users like you by:\n\n1. Answering questions on various topics, from science and history to entertainment and culture.\n2. Providing information and explanations on complex topics.\n3. Generating text based on a prompt or topic.\n4. Engaging in conversations and discussions.\n5. Offering suggestions and ideas on creative projects.\n\nI'm constantly learning and improving my responses based on user interactions, so feel free to ask me anything, and I'll do my b

In [21]:
groq_chat("what is your age ?")

'I was released to the public in 2023.'

# PART 2 - Groq + RAG

## Task 3 : Groq Based RAG Pipeline

In [ ]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_openai import OpenAI
from langchain_chroma import Chroma
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from dotenv import load_dotenv
load_dotenv()

USER_AGENT environment variable not set, consider setting it to identify your requests.


True

In [61]:
loader = WebBaseLoader('https://www.amazon.in/dp/B0DSKL9MQ8?_encoding=UTF8&ref_=cct_cg_Budget_2b1&pf_rd_p=4c7d86e9-7005-4b46-99de-a03abbb7c819&pf_rd_r=0QVP772NK55ZAJ83DJ98&th=1')

In [62]:
doc = loader.load()

In [63]:
splitter = RecursiveCharacterTextSplitter(chunk_size=2000,chunk_overlap=500)

In [64]:
chunks = splitter.split_documents(doc)

In [65]:
# chunks

In [66]:
len(chunks)

47

In [67]:
embeddings = OpenAIEmbeddings()

In [68]:
vector = Chroma.from_documents(documents=chunks,embedding=embeddings)

In [69]:
retriver = vector.as_retriever()

In [70]:
from langchain_core.prompts import ChatPromptTemplate,PromptTemplate

In [71]:
systemPrompt = '''you are an Helpful Question Answering AI System. 
                Answer the query from the provided context. If you don't know the answer directly say thank you and 
                I don't know the answer. Keep the answer concise. {context}'''

In [72]:
question="What is the price of the Phone ?"

In [73]:
prompt = ChatPromptTemplate([('system',systemPrompt),('human',question)])

In [74]:
from langchain_groq import ChatGroq

In [75]:
llm = ChatGroq(model='llama-3.1-8b-instant')

In [76]:
QAChain = create_stuff_documents_chain(llm,prompt)

In [77]:
rag = create_retrieval_chain(retriver,QAChain)

In [81]:
result = rag.invoke({'input':question})

In [82]:
result

{'input': 'What is the price of the Phone ?',
 'context': [Document(id='ff0d20ef-74ab-4299-a93e-5fe88a6a002f', metadata={'title': 'Samsung Galaxy S25 Ultra 5G Mobile with Galaxy AI', 'source': 'https://www.amazon.in/dp/B0DSKL9MQ8?_encoding=UTF8&ref_=cct_cg_Budget_2b1&pf_rd_p=4c7d86e9-7005-4b46-99de-a03abbb7c819&pf_rd_r=0QVP772NK55ZAJ83DJ98&th=1', 'language': 'en-in', 'description': 'Samsung Galaxy S25 Ultra 5G AI Smartphone (Titanium Whitesilver, 12GB RAM, 256GB Storage), 200MP Camera, S Pen Included, Long Battery Life : Amazon.in: Electronics'}, page_content='Would you like to  \n tell us about a lower price?\n                     \n \n \n\n\n\n\n\n\n\nSamsung Galaxy S25 Ultra 5G AI Smartphone (Titanium Whitesilver, 12GB RAM, 256GB Storage), 200MP Camera, S Pen Included, Long Battery Life\n\nShare:\n                                    \n\n\n\n\n\n\nFound a lower price? Let us know.\nWhere did you see a lower price?\nFields with an asterisk * are required\n\n\nPrice Availability\n\n\n\